In [2]:
import yfinance as yf
import pandas as pd
import sqlalchemy 
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [3]:
load_dotenv("../../.env")

mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_password = os.environ.get("MYSQL_PASSWORD")
mysql_database = os.environ.get("MYSQL_DATABASE")


#the f goes in front to embed variables
engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_database}")

with engine.connect() as conn:
    print("Connection successful")

# os.getcwd()

Connection successful


In [3]:
tickers = yf.Tickers('EQIX DLR IRM')

raw_data = yf.download("EQIX DLR IRM", period = '1y')

raw_data.head()


# The table intially started in a multilayered column format. The following code adjusts it to row format to match the format of the MYSQL databases
raw_pivot = raw_data.stack([0,1])
raw_pivot = raw_pivot.unstack(1)
raw_pivot = raw_pivot.reset_index([0,1])
raw_pivot.columns.name = None

#renaming the columns to match the sql schema
raw_pivot = raw_pivot.rename(columns={
    "Close": "adjusted_close",
    "High": "high",
    "Low": "low",
    "Open": "open",
    "Volume": "volume",
    "Date": "date",
    "Ticker": "ticker"
})
raw_pivot["volume"] = raw_pivot["volume"].astype(int)

raw_pivot['previous_return'] = raw_pivot.groupby('ticker')['adjusted_close'].shift(1)
raw_pivot['daily_return'] = (raw_pivot['adjusted_close'] - raw_pivot['previous_return']) / raw_pivot['previous_return']
raw_pivot = raw_pivot.drop(columns=['previous_return'])

print(raw_pivot.head)
raw_pivot.dtypes

[*********************100%***********************]  3 of 3 completed

<bound method NDFrame.head of           date ticker  adjusted_close         high          low         open  \
0   2025-06-12    DLR      171.527008   173.033083   170.535911   170.720529   
1   2025-06-12   EQIX      876.485657   882.430468   873.620820   873.826172   
2   2025-06-12    IRM       99.038788    99.270659    97.647547    97.744159   
3   2025-06-13    DLR      170.705124   171.057356   168.953742   170.274616   
4   2025-06-13   EQIX      872.789734   874.461669   863.667195   872.750585   
..         ...    ...             ...          ...          ...          ...   
751 2026-06-11   EQIX     1043.180054  1048.229980  1032.079956  1041.199951   
752 2026-06-11    IRM      125.169998   125.790001   122.720001   124.160004   
753 2026-06-12    DLR      183.300003   185.440002   183.300003   183.910004   
754 2026-06-12   EQIX     1046.859985  1055.614990  1043.750000  1048.780029   
755 2026-06-12    IRM      126.665001   127.949997   125.565002   125.739998   

      vol


X:\Temp\ipykernel_28148\1929844489.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  raw_pivot = raw_data.stack([0,1])


date              datetime64[ns]
ticker                    object
adjusted_close           float64
high                     float64
low                      float64
open                     float64
volume                     int64
daily_return             float64
dtype: object

In [4]:
#This is writing the company info to the companies table

company_df = pd.DataFrame({ 
    "ticker": ['EQIX', 'IRM', 'DLR'],
    "company_name": ["Equinix", "Iron Mountain", 'Digital Reality Trust'],
    "sector": ["REIT", "REIT", "REIT"],
    "exchange": ["NYSE", "NYSE", "NYSE"]
})

company_df.to_sql(name = 'companies', con = engine, if_exists = 'append', index = False)


IntegrityError: (mysql.connector.errors.IntegrityError) 1062 (23000): Duplicate entry 'EQIX' for key 'companies.PRIMARY'
[SQL: INSERT INTO companies (ticker, company_name, sector, exchange) VALUES (%(ticker)s, %(company_name)s, %(sector)s, %(exchange)s)]
[parameters: [{'ticker': 'EQIX', 'company_name': 'Equinix', 'sector': 'REIT', 'exchange': 'NYSE'}, {'ticker': 'IRM', 'company_name': 'Iron Mountain', 'sector': 'REIT', 'exchange': 'NYSE'}, {'ticker': 'DLR', 'company_name': 'Digital Reality Trust', 'sector': 'REIT', 'exchange': 'NYSE'}]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [5]:
raw_pivot.to_sql(name = 'daily_prices', con = engine, if_exists = 'append', index = False)

756

In [5]:
#pulling live data from yfinance
import nest_asyncio
import asyncio

nest_asyncio.apply()


# define your message callback
def message_handler(message):   
    ticker = message["id"]
    price = message["price"]
    with engine.begin() as conn:
        conn.execute(
            sqlalchemy.text("UPDATE live_prices SET current_price = :price WHERE ticker = :ticker"),
            {"price": price, "ticker": ticker} #this defines what the variables in the line above mean
        )
    print("Received message:", message)

async def main():
    # =======================
    # With Context Manager
    # =======================
    async with yf.AsyncWebSocket() as ws:
        await ws.subscribe(["DLR", "EQIX", "IRM"])
        await ws.listen(message_handler)

    # # =======================
    # # Without Context Manager
    # # =======================
    # ws = yf.AsyncWebSocket()
    # await ws.subscribe(["DLR", "EQIX", "IRM"])
    # await ws.listen()

asyncio.run(main())

Connected to WebSocket.
Subscribed to symbols: ['DLR', 'EQIX', 'IRM']
Listening for messages...
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
Heartbeat subscription sent for symbols: {'IRM', 'DLR', 'EQIX'}
WebSocket listening interrupted. Closing connection...
WebSocket connection closed.
WebSocket connection closed.


KeyboardInterrupt: 

In [ ]:


while True:
    for ticker in live_prices['ticker'].unique():
        temp_df[ticker] = live_prices[live_prices['ticker'] == ticker]
        data = yf.download(ticker, period="1d")
        temp_df[ticker] = data['High', ticker].iloc[0]
        temp_df[ticker] = data['Low', ticker].iloc[0]
    time.sleep(600) 


temp_df[ticker] = yf.download("EQIX", period="1d")
        temp_df[ticker] = yf.download("DLR", period="1d")
        temp_df[ticker] = yf.download("IRM", period="1d")

yf.download("EQIX", period="1d")

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EQIX,EQIX,EQIX,EQIX,EQIX
Date,,,,,
2026-06-12,1048.824951,1055.61499,1043.75,1048.780029,169484


In [20]:
data = yf.download("EQIX", period="1d")
print(data["Open", "EQIX"].iloc[0])

[*********************100%***********************]  1 of 1 completed

1048.780029296875


In [4]:
import time
for ticker in ["EQIX", "DLR", "IRM"]:
    data = yf.download(ticker, period="1d")
    high = data["High", ticker].iloc[0]
    low = data["Low", ticker].iloc[0]
    with engine.begin() as conn:
        conn.execute(
            sqlalchemy.text("UPDATE live_prices SET current_high = :high, current_low = :low WHERE ticker = :ticker"),
            {"high": high, "low": low, "ticker": ticker}
        )
    time.sleep(5)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
